# 5단계 실험: LangSmith Trace 연결

환경변수만 켜면 LCEL 체인은 자동 추적됩니다. 외부 진입 함수 4개에는 `@traceable` 이 부착되어 있어 각 단계가 라벨링된 부모 span 으로 묶입니다.

**준비**
- LangSmith 계정에서 API 키 발급 → `.env` 의 `LANGSMITH_API_KEY` 채우기
- `LANGSMITH_TRACING=true`, `LANGSMITH_PROJECT=llm-jacky`
- LangSmith 대시보드: https://smith.langchain.com/

## 0. 환경 설정 + 추적 활성화 검증

**중요**: `LANGSMITH_*` 환경변수는 LangChain 모듈을 import 하기 _전에_ 세팅돼야 합니다. `load_dotenv()` 를 첫 import 전에 호출하세요.

In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "TAVILY_API_KEY",
         "LANGSMITH_TRACING", "LANGSMITH_API_KEY", "LANGSMITH_PROJECT"):
    val = os.getenv(k)
    show = "OK" if val else "MISSING"
    if k == "LANGSMITH_TRACING":
        show = val or "MISSING"
    if k == "LANGSMITH_PROJECT":
        show = val or "MISSING"
    print(f"{k}: {show}")

tracing_on = (os.getenv("LANGSMITH_TRACING", "").lower() == "true"
              and bool(os.getenv("LANGSMITH_API_KEY")))
print("\ntracing_on:", tracing_on)

OPENAI_API_KEY: OK
ANTHROPIC_API_KEY: OK
TAVILY_API_KEY: OK
LANGSMITH_TRACING: true
LANGSMITH_API_KEY: OK
LANGSMITH_PROJECT: llm-jacky

tracing_on: True


## 1. SDK 연결 헬스체크

API 키가 유효한지 LangSmith client 로 확인합니다 (호출량 미세함).

In [2]:
from langsmith import Client

client = Client()
try:
    project_name = os.getenv("LANGSMITH_PROJECT", "default")
    proj = client.read_project(project_name=project_name)
    print(f"project found: {proj.name} (id={proj.id})")
except Exception as e:
    print("project read failed (자동 생성될 수 있음):", e)
    print("-> 첫 trace 가 들어가면 프로젝트가 자동 생성됩니다.")

project found: llm-jacky (id=432151b4-aac0-45ce-8782-039c02717a73)


## 2. 풀 파이프라인 1회 실행 (research → rag → writing → seo)

각 단계는 LangSmith 에서 별도의 부모 span 으로 표시됩니다.

In [3]:
from llm_jacky.research import run_research
from llm_jacky.rag import index_research
from llm_jacky.writing import write_draft
from llm_jacky.seo import generate_seo

TOPIC = "외국인을 위한 한국 길거리 음식 추천"

research = run_research(TOPIC, max_results=4)
n_chunks = index_research(research)
draft = write_draft(TOPIC, k=4)
meta = generate_seo(TOPIC, draft.draft)

print(f"sources: {len(research.sources)} | indexed chunks: {n_chunks}")
print(f"draft chars: {len(draft.draft)}")
print(f"seo title: {meta.title}")

/Users/jackykim/project/llm-jacky/src/llm_jacky/research.py:68: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults(max_results=max_results)


sources: 4 | indexed chunks: 13
draft chars: 1850
seo title: 외국인을 위한 한국 길거리 음식 추천


## 3. Trace 확인하는 법

LangSmith 대시보드에서 프로젝트(`llm-jacky`) 를 열면 방금 실행한 4개 부모 run 이 보입니다.

- `research` — Tavily 호출 + 요약 LLM 호출
- `rag.index` — 임베딩 호출
- `writing` — retriever + Claude 호출
- `seo` — 구조화 출력 LLM 호출

각 run 에서 토큰 수/지연/입출력/에러를 확인할 수 있습니다.

In [4]:
project_url = f"https://smith.langchain.com/projects/p/{os.getenv('LANGSMITH_PROJECT','default')}"
print("대시보드:", "https://smith.langchain.com/")
print("프로젝트:", os.getenv('LANGSMITH_PROJECT','default'))

대시보드: https://smith.langchain.com/
프로젝트: llm-jacky


## 4. 메타데이터/태그로 trace 그룹화

실험 비교용으로 prompt 버전, 사용자 id 등을 메타로 붙여 필터링할 수 있습니다.

In [5]:
from langsmith.run_helpers import tracing_context

with tracing_context(tags=["prompt-v1", "experiment-streetfood"],
                     metadata={"author": "jacky", "k": 4}):
    draft_v1 = write_draft(TOPIC, k=4)

print("draft_v1 chars:", len(draft_v1.draft))
print("-> LangSmith 에서 tag/metadata 로 필터하면 이 run 만 골라볼 수 있음")

draft_v1 chars: 1856
-> LangSmith 에서 tag/metadata 로 필터하면 이 run 만 골라볼 수 있음


## 5. 비용/지연 sanity check

프로젝트 페이지의 통계 패널에서 토큰/달러/지연을 바로 봅니다. 코드로도 최근 run 을 끌어 비용을 합산할 수 있습니다.

In [6]:
from datetime import datetime, timedelta, timezone

since = datetime.now(timezone.utc) - timedelta(minutes=15)
runs = list(client.list_runs(
    project_name=os.getenv("LANGSMITH_PROJECT", "default"),
    start_time=since,
    is_root=True,
))
print(f"recent root runs: {len(runs)}\n")
for r in runs[:10]:
    name = r.name
    latency = (r.end_time - r.start_time).total_seconds() if r.end_time else None
    tokens = (r.total_tokens or 0)
    print(f"  {name:14s} | {latency!s:>6}s | {tokens:>6} tokens")

recent root runs: 5

  writing        |   Nones |      0 tokens
  seo            | 2.748795s |   1563 tokens
  writing        | 22.695604s |   5418 tokens
  rag.index      | 1.162901s |      0 tokens
  research       | 15.910072s |   6261 tokens
